# 01 — Data Exploration

Exploratory data analysis of the NinaPro DB2 and Hyser datasets.

This notebook loads raw EMG and force data, visualizes signal characteristics,
and verifies preprocessing steps before model training.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.config import (
    RAW_NINAPRO_DIR, NINAPRO_FS, NINAPRO_CHANNEL_NAMES,
    BANDPASS_LOW, BANDPASS_HIGH, BANDPASS_ORDER,
    ENVELOPE_WINDOW_MS, ENVELOPE_HOP_MS,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load NinaPro DB2 — Single Subject

In [ ]:
from src.data.load_ninapro import load_ninapro_subject

# Load subject 1
mat_path = str(RAW_NINAPRO_DIR / 'S1_E3_A1.mat')
subj = load_ninapro_subject(mat_path)

emg = subj['emg']
force = subj['force']

print(f"EMG shape: {emg.shape}  (samples x channels)")
print(f"Force shape: {force.shape}")
print(f"Duration: {len(emg) / NINAPRO_FS:.1f} seconds")
print(f"Sampling rate: {NINAPRO_FS} Hz")

## 2. Raw EMG Signals

In [ ]:
# Plot first 2 seconds of all 10 channels
t_show = 2  # seconds
n_show = int(t_show * NINAPRO_FS)
t = np.arange(n_show) / NINAPRO_FS

fig, axes = plt.subplots(5, 2, figsize=(16, 12), sharex=True)
for i, ax in enumerate(axes.flat):
    if i < emg.shape[1]:
        ax.plot(t, emg[:n_show, i], linewidth=0.3, color='steelblue')
        ax.set_ylabel(NINAPRO_CHANNEL_NAMES[i], fontsize=8)
        ax.grid(alpha=0.2)
axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')
fig.suptitle('Raw EMG — Subject 1 (first 2 seconds)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Force Signal

In [ ]:
# Plot total grip force
t_all = np.arange(len(force)) / NINAPRO_FS

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t_all, force, linewidth=0.5, color='darkorange')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Total Grip Force')
ax.set_title('Force Signal — Subject 1 (full recording)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Force range: [{force.min():.4f}, {force.max():.4f}]")
print(f"Force mean: {force.mean():.4f}")
print(f"Force std: {force.std():.4f}")

## 4. EMG Frequency Spectrum

In [ ]:
from scipy.signal import welch

# Power spectral density for channel 0
f, psd = welch(emg[:, 0], fs=NINAPRO_FS, nperseg=2048)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(f, psd, color='steelblue')
ax.axvline(BANDPASS_LOW, color='red', linestyle='--', alpha=0.7, label=f'Bandpass: {BANDPASS_LOW}-{BANDPASS_HIGH} Hz')
ax.axvline(BANDPASS_HIGH, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD')
ax.set_title(f'Power Spectral Density — {NINAPRO_CHANNEL_NAMES[0]}')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Bandpass Filtering Effect

In [ ]:
from src.data.preprocess import bandpass_filter

# Filter a single channel
raw_ch = emg[:, 0]
filtered_ch = bandpass_filter(raw_ch, BANDPASS_LOW, BANDPASS_HIGH, NINAPRO_FS, BANDPASS_ORDER)

n_show = int(0.5 * NINAPRO_FS)  # 0.5 seconds
t = np.arange(n_show) / NINAPRO_FS

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(t, raw_ch[:n_show], linewidth=0.5, color='gray')
ax1.set_ylabel('Raw EMG')
ax1.set_title('Before Bandpass Filter')
ax1.grid(alpha=0.3)

ax2.plot(t, filtered_ch[:n_show], linewidth=0.5, color='steelblue')
ax2.set_ylabel('Filtered EMG')
ax2.set_xlabel('Time (s)')
ax2.set_title(f'After Bandpass Filter ({BANDPASS_LOW}-{BANDPASS_HIGH} Hz)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. RMS Envelope

In [ ]:
from src.data.preprocess import compute_rms_envelope

# Compute RMS envelope for all channels
envelope = compute_rms_envelope(emg, ENVELOPE_WINDOW_MS, ENVELOPE_HOP_MS, NINAPRO_FS)

print(f"Envelope shape: {envelope.shape}")
print(f"Effective rate: {NINAPRO_FS / (ENVELOPE_HOP_MS * NINAPRO_FS / 1000):.1f} Hz")

# Plot envelope for 2 channels vs force
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

t_env = np.arange(len(envelope)) * ENVELOPE_HOP_MS / 1000
axes[0].plot(t_env, envelope[:, 0], color='steelblue', linewidth=0.5)
axes[0].set_ylabel(NINAPRO_CHANNEL_NAMES[0])
axes[0].set_title('RMS Envelope')
axes[0].grid(alpha=0.3)

axes[1].plot(t_env, envelope[:, 9], color='steelblue', linewidth=0.5)
axes[1].set_ylabel(NINAPRO_CHANNEL_NAMES[9])
axes[1].grid(alpha=0.3)

t_force = np.arange(len(force)) / NINAPRO_FS
axes[2].plot(t_force, force, color='darkorange', linewidth=0.5)
axes[2].set_ylabel('Grip Force')
axes[2].set_xlabel('Time (s)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Channel Correlation with Force

In [ ]:
# Downsample force to match envelope rate
win_samp = int(ENVELOPE_WINDOW_MS * NINAPRO_FS / 1000)
n_windows = len(envelope)
force_env = np.array([np.mean(force[j*win_samp:(j+1)*win_samp]) for j in range(n_windows)])

# Compute Pearson correlation of each channel's envelope with force
correlations = []
for ch in range(envelope.shape[1]):
    r = np.corrcoef(envelope[:, ch], force_env)[0, 1]
    correlations.append(r)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(correlations)), correlations, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(correlations)))
ax.set_xticklabels(NINAPRO_CHANNEL_NAMES, rotation=45, ha='right')
ax.set_ylabel('Pearson Correlation')
ax.set_title('Channel-Force Correlation (RMS Envelope)')
ax.grid(axis='y', alpha=0.3)
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

# Print sorted
sorted_idx = np.argsort(correlations)[::-1]
print("Channels ranked by |correlation| with force:")
for i in sorted_idx:
    print(f"  {NINAPRO_CHANNEL_NAMES[i]:>25s}: {correlations[i]:.4f}")

## 8. Multi-Subject Statistics

In [ ]:
from src.data.load_ninapro import load_all_ninapro

# Load all subjects (may take a minute)
all_subjects = load_all_ninapro(str(RAW_NINAPRO_DIR))
print(f"Loaded {len(all_subjects)} subjects")

# Compute statistics
durations = [len(s['emg']) / NINAPRO_FS for s in all_subjects]
force_maxes = [s['force'].max() for s in all_subjects]
force_means = [s['force'].mean() for s in all_subjects]

print(f"\nRecording duration: {np.mean(durations):.1f} +/- {np.std(durations):.1f} s")
print(f"Max force: {np.mean(force_maxes):.2f} +/- {np.std(force_maxes):.2f}")
print(f"Mean force: {np.mean(force_means):.4f} +/- {np.std(force_means):.4f}")